In [ ]:
import fs from 'node:fs';
import path from 'node:path';
import { ChatOpenAI } from '@langchain/openai';
import { OPENAI_API_KEY } from './src/lib/vars.mjs';
import * as z from 'zod';

function logFile(logContent, fileName = 'jupyter.md') {
  const logFileName = `logs/${fileName}`;
  const logDir = path.dirname(logFileName);
  if (!fs.existsSync(logDir)) {
    fs.mkdirSync(logDir, { recursive: true });
  }
  fs.writeFileSync(logFileName, '```markdown\n' + logContent + '\n```', 'utf8');
  return logContent;
}

const gpt5 = new ChatOpenAI({
  modelName: 'gpt-5',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt4oMini = new ChatOpenAI({
  modelName: 'gpt-4o-mini',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});
const gpt5Nano = new ChatOpenAI({
  modelName: 'gpt-5-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 1,
});
const gpt41Nano = new ChatOpenAI({
  modelName: 'gpt-4.1-nano',
  apiKey: OPENAI_API_KEY,
  temperature: 0.1,
});

async function queryAllModels(prompt, callFunction) {
  console.log(`Started: ${new Date()}\n`);
  return Promise.all([
    callFunction(prompt, gpt4oMini),
    callFunction(prompt, gpt41Nano),
    callFunction(prompt, gpt5Nano),
    // callFunction(prompt, gpt5),
  ]);
}

## Rewrite Initial Query


### Sanitize Query

**Results:** 

All models performed satisfactory with similar times.

In [2]:
import { sanitizeQuery } from './src/lib/rag.mjs';

const redditQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

async function sanitize(question, model) {
  const { answer: query } = await sanitizeQuery(question, model);
  console.log(`**${model.model}** – ${new Date()}:\n${query}\n`);
  return query;
}

await queryAllModels(redditQuestion, sanitize);


Started: Sat Aug 23 2025 01:54:28 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Sat Aug 23 2025 01:54:30 GMT-0400 (Eastern Daylight Time):
I am an exchange student planning to study in Canada for a year, but I am concerned about my visa application. We submitted the visa request around June 15, and were initially told it would take about 6 weeks, but now the processing time has changed to 10 weeks. I need to leave by August 29 to start school on September 2, and I am worried that my visa will not arrive in time. We are considering applying for a tourist visa to enter Canada while waiting for my study visa, but I am concerned that this may affect my ability to return to Canada after Christmas. What should I do in this situation regarding my visa options and travel plans?

**gpt-5-nano** – Sat Aug 23 2025 01:54:30 GMT-0400 (Eastern Daylight Time):
I am an exchange student planning to study in Canada for a year, but I am concerned about my visa application timeline. We submitted th

[
  "I am an exchange student planning to study in Canada for a year, but I am concerned about my visa application. We submitted the visa request around June 15, and were initially told it would take about 6 weeks, but the processing time has now changed to 10 weeks. I need to leave by August 29, as school starts on September 2, and I cannot arrive late. Given the low likelihood of receiving my visa by the 29th, we are considering applying for a tourist visa to enter Canada and then receive my study visa later. However, I am worried that using a tourist visa might affect my ability to return to Canada after Christmas. What should I do regarding my visa situation?",
  "I am an exchange student planning to study in Canada for a year, but I am concerned about my visa application. We submitted the visa request around June 15, and were initially told it would take about 6 weeks, but now the processing time has changed to 10 weeks. I need to leave by August 29 to start school on September 2,

### Query Extraction

#### Decomposing Main Query

#### Test Results

| Model          | Response Time | Performance | Notes                                                   |
| -------------- | ------------- | ----------- | ------------------------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        |                                                         |
| GPT-4.1-nano   | < 5 sec       | Good        |                                                         |
| GPT-5-nano     | < 30 sec      | Mediocre    | Questions need further break down.                      |
| GPT-5          | > 1 min       | Ok          | Questions well-thought but must be broken down further. |


In [3]:
const compoundQuestion = `Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to read and i cannot explain myself completely bc english isnt my first language, but help what should we do? (Ask any question you need to ask)`;

const questionExtractionPrompt = `You are an assistant that prepares user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration rules.
The user may provide a long, informal story or question. Your task is:
1. Identify all explicit and implicit questions they are asking.  
2. Rewrite each one as a clear, self-contained question that could be answered directly from IRCC documentation.  
3. Condense the result into the *smallest possible set of non-overlapping, atomic questions* that fully capture the user’s intent.  
4. Eliminate redundancy — avoid rephrasing the same issue multiple times.  
5. Do not provide answers — only the minimal list of questions.

**User question:**

\`\`\`
${compoundQuestion}
\`\`\`
`;

async function extractQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z
          .array(z.string())
          .describe('A list of questions derived from the user query, sorted by relevance top to bottom.'),
      })
    )
    .invoke(q);
  const content = response.questions.map((q) => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

const extractedQuestions = await queryAllModels(questionExtractionPrompt, extractQuestions);


Started: Sat Aug 23 2025 01:54:30 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** Sat Aug 23 2025 01:54:31 GMT-0400 (Eastern Daylight Time):
	- Can an exchange student enter Canada on a temporary resident visa (visitor visa) and then switch to a study permit after arrival?
	- What are the rules and restrictions for a visitor visa holder who wants to study in Canada?
	- Is it possible to leave Canada and re-enter on a visitor visa after starting a study program?
	- What are the implications of entering Canada on a visitor visa with respect to future re-entry and study permit applications?
	- Given the current processing times and deadlines, what are the recommended options for an exchange student planning to study in Canada for a year?

**gpt-4o-mini** Sat Aug 23 2025 01:54:33 GMT-0400 (Eastern Daylight Time):
	- What are the current processing times for a study permit application for exchange students in Canada?
	- Can I enter Canada on a tourist visa while waiting for my study perm

#### Identifying Key Questions

##### Test Summary

| Model           | Response Time | Performance | Notes                                  |
| --------------- | ------------- | ----------- | -------------------------------------- |
| 🥇 GPT-4.1-nano | < 5 sec       | Good        | Correctly identified the key question. |
| GPT-4o-mini     | < 5 sec       | Good        | Included some secondary questions.     |
| GPT-5-nano      | < 30 sec      | Mediocre    | Included the most questions.           |
| GPT-5           | < 30 sec      | Ok          | Included some secondary questions.     |


In [4]:

const mdExtractedQuestions = extractedQuestions[1].questions.map(q => `\t- ${q}`).join('\n');
const questionDiscriminationPrompt = `You are helping prepare user queries for a Retrieval-Augmented Generation (RAG) system about Canadian immigration.
Input: a list of atomic questions generated from a user’s long query.
Task:
1. Identify the key question(s) that directly capture the user’s main intent.  
   - Keep only the questions that must be answered to resolve the user’s core concern.  
   - Discard questions that are secondary, conditional, or only relevant as follow-ups.
2. Output only the minimal set of key questions, without explanation, ranked by relevance to the user query.
Important: The result should be as short as possible while still fully representing the original user’s primary intent.

User query:

\`\`\`
${compoundQuestion}
\`\`\`

List of questions:
\`\`\`
${mdExtractedQuestions}
\`\`\`
`

async function discriminateQuestions(q, model) {
  const response = await model
    .withStructuredOutput(
      z.object({
        questions: z.array(z.string()).describe('A list of key questions derived from the user query'),
      })
    )
    .invoke(q);
  const content = response.questions.map(q => `\t- ${q}`).join('\n');
  console.log(`**${model.model}** ${new Date()}:\n${content}\n`);
  return response;
}

console.log(`\n\nDiscriminating questions for: "${compoundQuestion}"\n`);
console.log(`Extracted questions:\n${mdExtractedQuestions}\n`);
await queryAllModels(questionDiscriminationPrompt, discriminateQuestions);



Discriminating questions for: "Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if th

[
  {
    questions: [
      "Can an exchange student enter Canada on a temporary resident visa (visitor visa) and then switch to a study permit after arrival?",
      "What are the implications of entering Canada on a visitor visa with respect to future re-entry and study permit applications?",
      "Given the current processing times and deadlines, what are the recommended options for an exchange student planning to study in Canada for a year?"
    ]
  },
  {
    questions: [
      "Given the current processing times and deadlines, what are the recommended options for an exchange student planning to study in Canada for a year?"
    ]
  },
  {
    questions: [
      "Given the current processing times and deadlines, what are the recommended options for an exchange student planning to study in Canada for a year?",
      "Can an exchange student enter Canada on a visitor visa and then switch to a study permit after arrival?",
      "What are the rules and restrictions for a visitor vis

## Vector Search

### Retrieval

In [5]:
import { vectorSearch, chunksToMarkdown } from './src/lib/vector-search.mjs';
const retrieveQuery = 'Can I enter Canada on a tourist visa while waiting for my study permit?'
const chunks = await vectorSearch(retrieveQuery);
logFile(chunksToMarkdown(chunks), 'chunks.md');
chunks

[
  {
    text: "# Study permit\n" +
      "\n" +
      "\\[...\\]\n" +
      "\n" +
      "* * * \n" +
      "\n" +
      "If you’re a lawful permanent resident of the United States, travel with a valid green card (or [equivalent official proof of status in the US](/en/immigration-refugees-citizenship/services/visit-canada/entry-requirements-country.html#lawful-pr-us)) and a valid passport from your country of nationality (or an equivalent document).\n" +
      "\n" +
      "### If you applied for your study permit from within Canada\n" +
      "\n" +
      "If you were eligible to apply from within Canada, you filled out a form called “Application to change conditions, extend my stay or remain in Canada as a student” (IMM 5709).\n" +
      "\n" +
      "In this case, we’ll mail the study permit to the Canadian address you gave us.\n" +
      "\n" +
      "If your mailing address changes before we send you our decision, you must [give us your new address](https://www.ircc.canada.ca/en

### Reference Discrimination

#### Test Summary

| Model          | Response Time | Performance | Notes                                 |
| -------------- | ------------- | ----------- | ------------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Picked the most chunks, all relevant. |
| GPT-4.1        | < 5 sec       | Ok          |                                       |
| GPT-5          | < 1 min       | Ok          |                                       |
| GPT-5-nano     | < 30 sec      | Mediocre    | Missed a highly relevant chunk.       |


In [ ]:
const chunkDiscriminationPrompt = `I'll give you a markdown content with a list of results from a vector search for a question.
Select only the references that help to answer th question.
Pay attention to the header of the top-most header in each reference to identify if it's related to the topic we want to answer; discard it if it is not.
Return an array containing the selected references' numbers.

**Question:** ${retrieveQuery}

**Chunks:**

\`\`\`markdown
${chunksToMarkdown(chunks)}
\`\`\`
`;

async function evaluateFollowUp(prompt, model) {
  const { references: referenceIndexes } = await model
  .withStructuredOutput(
    z.object({
      references: z.array(z.number()).describe('An array of numbers representing the selected references from the chunks'),
    })
  )
  .invoke(prompt);

  console.log(`**${model.model}** – ${new Date()}: ${JSON.stringify(referenceIndexes)}`);
  return referenceIndexes;
}

const referenceIndexes = await queryAllModels(chunkDiscriminationPrompt, evaluateFollowUp);

Started: Sat Aug 23 2025 01:56:03 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Sat Aug 23 2025 01:56:04 GMT-0400 (Eastern Daylight Time): [4]
**gpt-4o-mini** – Sat Aug 23 2025 01:56:04 GMT-0400 (Eastern Daylight Time): [4,5]
**gpt-5-nano** – Sat Aug 23 2025 01:56:13 GMT-0400 (Eastern Daylight Time): [4,5]
**gpt-5** – Sat Aug 23 2025 01:56:38 GMT-0400 (Eastern Daylight Time): [4,5]


## Answering

### Generating Answer

#### Test Summary

| Model          | Response Time | Performance | Notes                             |
| -------------- | ------------- | ----------- | --------------------------------- |
| 🥇 GPT-4o-mini | < 5 sec       | Good        | Concise and accurate.             |
| GPT-5-nano     | < 30 sec      | Good        | Good structure. Too many details. |
| GPT-4.1        | < 5 sec       | Mediocre    | Rarely provides sources.          |
| GPT-5          | < 1 min       | Ok          | Good structure. Too many details. |


In [9]:
import { generateAnswer } from './src/lib/rag.mjs';

async function generateRAGAnswer(query, model) {
  const selectChunks = chunks.filter((_, i) => referenceIndexes[1].includes(i + 1));
  const references = chunksToMarkdown(selectChunks);
  const response = await generateAnswer(query, references, model);
  console.log(
    `**${model.model}** – ${new Date()}:\n${response}\n\n* * *\n`
  );
  return response;
}

console.log(`"${retrieveQuery}"\n`);
const generatedResponses = await queryAllModels(retrieveQuery, generateRAGAnswer);

"Can I enter Canada on a tourist visa while waiting for my study permit?"

Started: Sat Aug 23 2025 02:04:46 GMT-0400 (Eastern Daylight Time)

**gpt-4.1-nano** – Sat Aug 23 2025 02:04:48 GMT-0400 (Eastern Daylight Time):
Yes, you can enter Canada on a visitor (tourist) visa while waiting for your study permit, provided you meet the requirements for entry as a visitor. When arriving at the port of entry, you should inform the officer that you intend to enter as a visitor. The officer will assess your eligibility, including your travel documents and medical requirements, and decide whether to permit entry as a visitor. If your study permit application is still being processed, entering as a visitor is an option, but it does not automatically grant you the right to study in Canada; you would need to obtain a study permit to study legally. Additionally, you cannot apply for a study permit at the port of entry; the application must be submitted beforehand or through other authorized channel

## Evaluating Answer

### Test Summary
...

In [10]:
const answer = generatedResponses[0];
const evaluationPrompt = `Evaluate if the answer provided by a RAG bot is fully addressing the user's concerns.

Input: 
1. The user's original question.
2. The answer generated by the RAG bot.

Your task:
- Determine if the answer fully addresses the user's question.
- If the answer is incomplete, does not address the user's concerns, or is irrelevant, return an array with a single required follow up questions to ask the RAG bot.
- The questions must be ordered by relevance, with the most important question first.
- If the answer is complete and directly addresses the user's concerns, return an empty array.

**User's question:**

\`\`\`
${redditQuestion}
\`\`\`

**RAG bot's answer:**

\`\`\`
${answer}
\`\`\`
`;

async function evaluateFollowUp(prompt, model) {
  const { questions } = await model
  .withStructuredOutput(
    z.object({
      questions: z.array(z.string()).describe('An array containing 1 question as input to the RAG bot'),
    })
  )
  .invoke(prompt);

  console.log(`**${model.model}** – ${new Date()}: ${JSON.stringify(questions, null, 2)}\n\n`);
  return questions;
}

console.log(`—"${redditQuestion}"\n\n—"${answer}"\n\n* * *\n\n`);
const followUpQuestions = await queryAllModels(evaluationPrompt, evaluateFollowUp);

—"Okay help, i need advice, i’m an exchange student who’s planning to go study to Canada for the whole year, unfortunately, we started doing the visa request way too late, maybe on 15th of june or smth like that ( maybe a bit earlier) and they said they were gonna take around 6 weeks, they now changed the policy to 10 but they assured us it would take 6 weeks. We have been planning on me to leave the 29th of august because its the best option , and school starts the 2nd of sep and i CANNOT arrive late. The possibilities of the visa arriving before the 29th are low and nobody can tell us when will it arrive. We are planning on to send me to Canada with a tourist visa and whenever they have my visa they can give it to me, the problem now is i wanna go back to my country in christmas and maybe if we do the tourist visa thing i might not be able to go back to Canada after christmas because of the fact that i first went there with a tourist one. Im sorry if this was too long or boring to re

In [29]:
let result = await gpt4oMini.invoke(`## Instructions  
You are a helpful and reliable RAG chatbot assistant.
Your task is to answer the user's question using only the information provided in the context ("IRCC documentation") below.
The context is a compilation of text chunks extracted from the IRCC documentation, each separated by a horizontal divider (---).
The references to the original documents are numbered and provided at the end of each chunk, just above the divider.

1. Structure your answer using the Pyramid Principle: start with a clear summary of the answer, followed by supporting details, and end with references.
2. Answer in Markdown format. Do not break the answer into multiple sections explicitly, but rather provide a single cohesive response.
3. Cite the source for argument using the 'Reference' provided at the end of each chunk.
4. Place the citation immediately after the relevant statement in this format: [[<reference number>](https://example.com)].
5. If the IRCC documentation does not contain a clear answer, say so honestly. Do not guess or fabricate information.
6. Be accurate, concise, and neutral in tone. 
7. Highlight any potential nuances in the answer that depend on the user's specific scenario and conditions.

## Question:

\`\`\`
Can I enter Canada on a tourist visa while my study permit application is being processed?
\`\`\`

## Context:

\`\`\`markdown
# Guide 5552 - Applying to Change Conditions or Extend Your Stay in Canada - Student - online application

[Print](javascript:window.print\(\);)

## You need a provincial attestation letter (PAL) or territorial attestation letter (TAL) to apply for a study permit

Most students must include with their study permit application a PAL/TAL from the province or territory where they plan to study.

In most cases, if you apply without a PAL/TAL, your application will be returned with fees.

[Learn more about the provincial attestation letter and territorial attestation letter](/en/immigration-refugees-citizenship/services/study-canada/study-permit/get-documents/provincial-attestation-letter.html).

## Updated application form for study permit extensions

\[...\]

* * * 

##### Travelling outside Canada:

**If you have applied to extend your study permit and plan to travel outside Canada while your application is in process,** you can leave and come back. However, one of two things will happen when you return to Canada:

*   You may be allowed to come back to Canada **as a visitor**, if Immigration, Refugees and Citizenship Canada (IRCC) has not yet made a decision on your study permit application. **If this is the case, you cannot study until you get your new study permit**. The officer at the port of entry may ask you to prove you have enough money to support yourself in Canada.
*   You may be allowed to come back to Canada as a student, if the officer at the port of entry determines that IRCC issued your study permit while you were away.

**Note:**

It is possible that you will not be able to enter Canada. The final decision is always made by the officer at the port of entry.

\[...\]

* * * 



Reference [1]: https://www.canada.ca/en/immigration-refugees-citizenship/services/application/application-forms-guides/guide-5552-applying-change-conditions-extend-your-stay-canada-student.html
-----------------------------
`);

JSON.stringify(result.content, null, 2);


'"{\\"questions\\":[\\"Can I enter Canada on a tourist visa while my study permit application is being processed?\\"]}"'